In [10]:
import importlib
import os
import pickle
from pathlib import Path

import numpy as np
import polars as pl 

from behave_analysis.visualize.visualize_utils import open_tracking_data
from behave_analysis.process.session import get_experiment
from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.analyze.single_trial import preprocess_regression
from behave_analysis.analyze.single_trial import single_trial_regression
from behave_analysis.analyze.single_trial.single_trial_regression import SingleTrialRegression
from behave_analysis.analyze.single_trial.preprocess_regression import PreprocessSingleTrialRegression
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr


# Sessions to consider

In [11]:
# JAL6
experiments_objects = [JAL6_28mar]

In [12]:
c_type = "good"
dir = make_directory(r"Z:\Jasmine_Laurence\single_trial_overview")
dir = Path(dir)

# Loop through sessions

In [13]:
importlib.reload(preprocess_regression)
importlib.reload(single_trial_regression)

# Lazy naming of sessions TODO - hardcode session names
session_names = ["JAL6_28th_March"]

for i, session in enumerate(experiments_objects):
    loaded_session = get_experiment(session)
    
    # Define paths
    video_and_spike_data_path = os.path.join(loaded_session.base_path, loaded_session.processed_path, "good_video_spike_count_df.parquet")
    homing_path = os.path.join(loaded_session.base_path, loaded_session.processed_path, "homings", "homings_obj.pkl")
    escape_path = os.path.join(loaded_session.base_path, loaded_session.processed_path, "escapes", "escapes_obj.pkl")

    # Load data
    try:
        video_df = pl.read_csv(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "\\" "full_video_dataframe.csv")
        with open(homing_path, "rb") as hf:
                homings_object = pickle.load(hf)
        video_and_spike_data = pl.read_parquet(video_and_spike_data_path)
        frame_by_cluster_matrix = np.load(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "\\" + "frame_by_" + c_type + "_cluster_matrix.npy")
        tracking_data = open_tracking_data(loaded_session)
        with open(escape_path, "rb") as ef:
                escape_object = pickle.load(ef)
        cluster_Ids = np.load(str(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "/" + c_type + "_cluster_Ids.npy"))
    
    except FileNotFoundError:
        print("One of the files was not found")
    
    finally:
        # Check if the homing object has the expected keys
        angle_data = homings_object.homing_angles_dic
        expected_keys = ["avg_pre_flip_head_angle", "avg_post_flip_head_angle", "avg_hsa"]
        assert all(key in angle_data.keys() for key in expected_keys), "The keys are not as expected in the homing angle data, please regenerate the homing object"

    # Run analysis
    pp_single_trial_obj = PreprocessSingleTrialRegression(
        video_df=video_df,
        homings_obj=homings_object,
        video_and_spike_data=video_and_spike_data,
        frame_by_cluster_matrix=frame_by_cluster_matrix,
        save_path=dir,
        velocity_data=tracking_data["avg_Velocity"],
        similar_homings=False,
        barrier_location=tracking_data["barrier_loc"],
        shelter_location=tracking_data["shelter_loc"],
        escape_object=escape_object,
        remove_escapes=False,
        save_plots=False)
                    
    SingleTrialRegression(
        design_matrix=pp_single_trial_obj.design_matrix,
        save_path=dir,
        dependents_df=pp_single_trial_obj.targets_df,
        tracking_data=tracking_data,
        homing_list=pp_single_trial_obj.homing_list,
        spike_homing_list=pp_single_trial_obj.spike_data_per_homing,
        condition_per_homing=pp_single_trial_obj.condition_per_homing,
        cluster_ids=cluster_Ids,
        initial_directions=pp_single_trial_obj.initial_directions,
        conversion_from_left_right_to_pre_post_flip=pp_single_trial_obj.convert_left_right_to_pre_post_flip)
            

2024-08-20 18:13:56.032 | INFO     | behave_analysis.analyze.single_trial.preprocess_regression:__init__:53 - Initializing the single trial regression preprocessing object
2024-08-20 18:13:56.033 | INFO     | behave_analysis.utils.label_barrier_edges:check_which_barrier_location_is_which_orientation:15 - The barrier location pre flip is [227, 517] and post flip is [809, 516]
2024-08-20 18:13:56.034 | INFO     | behave_analysis.utils.label_barrier_edges:check_which_barrier_location_is_which_orientation:17 - The barrier preflip location is the left edge
2024-08-20 18:13:56.259 | INFO     | behave_analysis.utils.label_barrier_edges:check_which_barrier_location_is_which_orientation:15 - The barrier location pre flip is [227, 517] and post flip is [809, 516]
2024-08-20 18:13:56.261 | INFO     | behave_analysis.utils.label_barrier_edges:check_which_barrier_location_is_which_orientation:17 - The barrier preflip location is the left edge
2024-08-20 18:13:56.267 | SUCCESS  | behave_analysis.ana

TypeError: unsupported operand type(s) for +: 'WindowsPath' and 'str'